# Part A1 - The Onboarding Funnel

**What this notebook does:** builds the signup → approved funnel the brief asks for in **A1**, states our denominator and cohorting rule out loud, and shows exactly where captains are lost - not just percentages, but real counts. Every number here is calculated live in the code below it, nothing is typed in by hand.

### Step 0 - Load the tools
We load pandas (our data tool) and `common.py` - the shared rulebook every notebook in this project uses, so numbers can never quietly disagree between files.

In [4]:
import pandas as pd
import common

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

DATA_DIR = common.find_data_dir()
print(f"Setup complete. Data dir: {DATA_DIR.name}")
print(f"Snapshot: {common.SNAPSHOT_TS} | Mature cutoff: {common.MATURE_CUTOFF}")

Setup complete. Data dir: Rapido Assignment
Snapshot: 2026-06-30 23:59:00 | Mature cutoff: 2026-05-31 23:59:00


### Step 1 - Load the raw files
Reads the 5 files needed for this part, exactly as given, no edits. This is the raw material behind **A2O** and **R2A**, the two numbers the brief asks us to focus on.

In [5]:
captains, doc_events, approvals, activation, nudges = common.load_all(DATA_DIR)
print(f"captains:   {len(captains):,} rows")
print(f"doc_events: {len(doc_events):,} rows")
print(f"approvals:  {len(approvals):,} rows")
print(f"activation: {len(activation):,} rows")
print(f"nudges:     {len(nudges):,} rows")

captains:   25,000 rows
doc_events: 186,282 rows
approvals:  25,000 rows
activation: 4,206 rows
nudges:     16,314 rows


### The cohort rule (our denominator) - and proof it's correct
The brief asks us to state our **denominator and cohorting rule**, and say why. We only score someone as approved or not once they've had enough time to finish (30 days). This check proves that rule is correct: everyone older than 30 days already has a final answer.

In [17]:
master_check = captains.merge(approvals, on="captain_id", how="left")
for window in range(10, 25):
    cutoff = common.SNAPSHOT_TS - pd.Timedelta(days=window)
    mature_check = master_check[master_check["signup_ts"] < cutoff]
    in_prog = (mature_check["final_status"] == "in_progress").sum()
    marker = "  <- first window where backlog fully clears" if in_prog == 0 and window == min(
        w for w in range(10, 25)
        if (master_check[master_check["signup_ts"] < common.SNAPSHOT_TS - pd.Timedelta(days=w)]["final_status"] == "in_progress").sum() == 0
    ) else ""
    print(f"window={window:2d} days -> in_progress count={in_prog}{marker}")

print(f"\nUsing a 30-day window as a conservative buffer to avoid counting recent signups as drop-offs..")

captains["cohort_mature"] = captains["signup_ts"] < common.MATURE_CUTOFF
print(f"\nCaptains signed up before cutoff (mature): {captains['cohort_mature'].sum():,}")
print(f"Captains signed up after cutoff (recent):   {(~captains['cohort_mature']).sum():,}")

window=10 days -> in_progress count=69
window=11 days -> in_progress count=43
window=12 days -> in_progress count=16
window=13 days -> in_progress count=4
window=14 days -> in_progress count=2
window=15 days -> in_progress count=1
window=16 days -> in_progress count=0  <- first window where backlog fully clears
window=17 days -> in_progress count=0
window=18 days -> in_progress count=0
window=19 days -> in_progress count=0
window=20 days -> in_progress count=0
window=21 days -> in_progress count=0
window=22 days -> in_progress count=0
window=23 days -> in_progress count=0
window=24 days -> in_progress count=0

Using a 30-day window as a conservative buffer to avoid counting recent signups as drop-offs..

Captains signed up before cutoff (mature): 20,207
Captains signed up after cutoff (recent):   4,793


### Sanity check - do our two data sources agree?
The brief warns nothing here has been cleaned for us. Before trusting any number, we check that document records and approval records tell the same story. They do, once we set aside the very newest sign-ups.

In [18]:
passed_doc_count = (
    doc_events[doc_events["event_type"] == "verification_pass"]
    .groupby("captain_id")["doc_type"].nunique().rename("docs_passed_from_events")
)
stage_check = approvals.merge(passed_doc_count, on="captain_id", how="left")
stage_check["docs_passed_from_events"] = stage_check["docs_passed_from_events"].fillna(0)
mismatch = stage_check[stage_check["docs_cleared"] != stage_check["docs_passed_from_events"]]
print(f"Mismatched captains: {len(mismatch)} out of {len(approvals):,} ({len(mismatch)/len(approvals)*100:.1f}%)")

signup_month_lookup = captains.set_index("captain_id")["signup_ts"].dt.to_period("M").astype(str)
print("Signup month of every mismatched captain:")
print(mismatch["captain_id"].map(signup_month_lookup).value_counts())

mature_ids = set(captains[captains["cohort_mature"]]["captain_id"])
mismatch_in_mature = mismatch[mismatch["captain_id"].isin(mature_ids)]
print(f"\nMismatched captains ALSO in the mature cohort: {len(mismatch_in_mature)}")
print("(All mismatches are June signups -- an extraction-timing gap between the two")
print("files, not a data error. Zero mismatches survive once June is excluded.)")

Mismatched captains: 451 out of 25,000 (1.8%)
Signup month of every mismatched captain:
captain_id
2026-06    451
Name: count, dtype: int64

Mismatched captains ALSO in the mature cohort: 0
(All mismatches are June signups -- an extraction-timing gap between the two
files, not a data error. Zero mismatches survive once June is excluded.)


### Step 2 - Put everything into one table
Joins sign-up info, approval outcome, and first-ride activity into a single table, one row per captain, so we can measure the whole journey - A2O and R2A - at once.

In [8]:
master, master_mature, master_recent = common.build_master(captains, approvals, activation)
print(f"master (all):    {len(master):,} rows")
print(f"master_mature:   {len(master_mature):,} rows  <- used for headline funnel")
print(f"master_recent:   {len(master_recent):,} rows  <- excluded, reported separately")

master (all):    25,000 rows
master_mature:   20,207 rows  <- used for headline funnel
master_recent:   4,793 rows  <- excluded, reported separately


### Proof - the document order is really followed
Checks that nobody skips ahead (for example, clears Aadhaar without first clearing the Driving Licence). Confirms this is a real step-by-step **funnel**, not just six separate pass rates.

In [9]:
passed = doc_events[doc_events["event_type"] == "verification_pass"]
def passed_set(doc_type):
    return set(passed[passed["doc_type"] == doc_type]["captain_id"])

dl, rc, aadhaar, permit, fitness, insurance = (
    passed_set("DL"), passed_set("RC"), passed_set("AADHAAR"),
    passed_set("PERMIT"), passed_set("FITNESS"), passed_set("INSURANCE"),
)
autocab_ids = set(captains[captains["vehicle_type"].isin(["Auto", "Cab"])]["captain_id"])
erickshaw_ids = set(captains[captains["vehicle_type"] == "ERickshaw"]["captain_id"])

print("RC passed but DL NOT passed:              ", len(rc - dl))
print("AADHAAR passed but RC NOT passed:         ", len(aadhaar - rc))
print("PERMIT passed but AADHAAR NOT passed:     ", len(permit - aadhaar))
print("FITNESS passed but PERMIT NOT passed\n  (Auto/Cab only, who require Permit):  ", len((fitness & autocab_ids) - permit))
print("FITNESS passed but AADHAAR NOT passed\n  (ERickshaw only, who skip Permit):    ", len((fitness & erickshaw_ids) - aadhaar))
print("INSURANCE passed but FITNESS NOT passed:  ", len(insurance - fitness))
print("\nAll six transitions checked, all zero -> confirmed: strictly sequential funnel.")

RC passed but DL NOT passed:               0
AADHAAR passed but RC NOT passed:          0
PERMIT passed but AADHAAR NOT passed:      0
FITNESS passed but PERMIT NOT passed
  (Auto/Cab only, who require Permit):   0
FITNESS passed but AADHAAR NOT passed
  (ERickshaw only, who skip Permit):     0
INSURANCE passed but FITNESS NOT passed:   0

All six transitions checked, all zero -> confirmed: strictly sequential funnel.


### Step 3 - Build the funnel
This is the actual signup → approved funnel the brief asks for (**A1**). Shows counts at every stage, not just percentages, so it's clear exactly where people are lost.

In [10]:
funnel = common.build_funnel(master_mature, doc_events)
print(funnel.to_string(index=False))

biggest_drop_idx = funnel["pct_of_signups"].diff().idxmin()
print(f"\nBiggest single-stage drop: {funnel.loc[biggest_drop_idx-1,'stage']} -> {funnel.loc[biggest_drop_idx,'stage']}"
      f"  ({funnel.loc[biggest_drop_idx,'pct_of_signups']-funnel.loc[biggest_drop_idx-1,'pct_of_signups']:.1%} points)")

                                 stage  count  pct_of_signups  pct_of_eligible
                          1. Signed up  20207        1.000000              NaN
                         2. DL cleared  17895        0.885584         0.885584
                         3. RC cleared  13008        0.643737         0.643737
                    4. AADHAAR cleared  11631        0.575593         0.575593
     5. PERMIT cleared (Auto/Cab only)   7059        0.349334         0.431664
                    6. FITNESS cleared   6867        0.339833         0.339833
                  7. INSURANCE cleared   3888        0.192409         0.192409
          8. All required docs cleared   3888        0.192409         0.192409
9. Passed eligibility/background check   3532        0.174791         0.908436
           10. Approved (A2O complete)   3532        0.174791         1.000000

Biggest single-stage drop: 2. DL cleared -> 3. RC cleared  (-24.2% points)


### Proof - rejection happens after paperwork, not instead of it
Confirms every rejected captain had already cleared all their documents. This tells us rejection is a separate, later gate — not a document problem.

In [11]:
rejected_mature = master_mature[master_mature["final_status"] == "rejected"]
fully_cleared_rejected = rejected_mature[rejected_mature["docs_cleared"] >= rejected_mature["required_docs_n"]]
print(f"Rejected captains: {len(rejected_mature)}")
print(f"Of those, cleared all required docs first: {len(fully_cleared_rejected)}  "
      f"({len(fully_cleared_rejected)/len(rejected_mature):.1%})")

excess = master_mature[master_mature["docs_cleared"] > master_mature["required_docs_n"]]
print(f"\nCaptains where docs_cleared > required_docs_n (would make >= and == differ): {len(excess)}")
print("-> >= and == are equivalent in this data; >= is used defensively in build_funnel.")

Rejected captains: 356
Of those, cleared all required docs first: 356  (100.0%)

Captains where docs_cleared > required_docs_n (would make >= and == differ): 0
-> >= and == are equivalent in this data; >= is used defensively in build_funnel.


### Step 4 - Why are captains actually lost?
This is the start of **A2 - finding the biggest fixable leak**. Sorts every lost captain into a plain reason: failed and gave up, never failed but went quiet, never started, or turned down at the end.

In [12]:
doc_events_mature = doc_events[doc_events["captain_id"].isin(master_mature["captain_id"])]
result = common.classify_doc_loss_reason(master_mature, doc_events_mature)
print(result["loss_bucket"].value_counts())
print("\n(Reconciliation check already ran inside classify_doc_loss_reason -- would have")
print("raised an AssertionError here if the bucket totals didn't add up.)")

loss_bucket
failed_gave_up        8887
stalled_no_failure    5967
approved              3532
never_started         1248
post_doc_rejection     356
exhausted_attempts     217
Name: count, dtype: int64

(Reconciliation check already ran inside classify_doc_loss_reason -- would have
raised an AssertionError here if the bucket totals didn't add up.)


### Which document causes the most permanent drop-out?
For each document, checks what share of the people who fail it ever come back to try again. Finds the single sharpest problem document.

In [13]:
retry_behavior = pd.DataFrame([common.post_failure_outcome(doc_events_mature, d) for d in common.DOC_ORDER])
print(retry_behavior.to_string(index=False))

rc_row = retry_behavior[retry_behavior["doc_type"] == "RC"].iloc[0]
next_highest = retry_behavior[retry_behavior["doc_type"] != "RC"]["captains_who_ever_failed"].max()
print(f"\nRC: {rc_row['captains_who_ever_failed']:,.0f} ever-failed -- "
      f"{rc_row['captains_who_ever_failed']/next_highest:.2f}x the next-highest document. "
      f"Never-retry rate: {rc_row['never_retried_pct']:.1%} -- essentially a coin flip.")

 doc_type  captains_who_ever_failed  never_retried_after_fail  never_retried_pct  eventually_passed_pct
       DL                      2690                      1064           0.395539               0.604461
       RC                      5465                      2727           0.498994               0.501006
  AADHAAR                      1167                       452           0.387318               0.612682
   PERMIT                      1394                       565           0.405308               0.594692
  FITNESS                      1986                       827           0.416415               0.583585
INSURANCE                      1493                       698           0.467515               0.532485

RC: 5,465 ever-failed -- 2.03x the next-highest document. Never-retry rate: 49.9% -- essentially a coin flip.


### Why does that document fail so often?
Looks at the real reasons that document fails. Confirms it's mostly bad photos, not a genuine paperwork issue — the kind of thing a product team could actually fix (the brief's "Monday test").

In [14]:
rc_fails = doc_events_mature[
    (doc_events_mature["doc_type"] == "RC") & (doc_events_mature["event_type"] == "verification_fail")
]
print(f"Total RC failure events (mature cohort): {len(rc_fails):,}")
print()
reason_shares = rc_fails["failure_reason"].value_counts(normalize=True)
print("RC failure_reason breakdown:")
print(reason_shares.round(3))

capture_quality_share = rc_fails["failure_reason"].isin(["image_blurred", "ocr_low_confidence"]).mean()
print(f"\nimage_blurred + ocr_low_confidence combined share: {capture_quality_share:.1%}")

Total RC failure events (mature cohort): 6,694

RC failure_reason breakdown:
failure_reason
image_blurred          0.279
ocr_low_confidence     0.218
name_mismatch          0.136
document_expired       0.113
details_not_legible    0.103
wrong_document_type    0.096
duplicate_document     0.056
Name: proportion, dtype: float64

image_blurred + ocr_low_confidence combined share: 49.7%


### Does phone quality explain it?
Checks the same failure rate split by phone type. This number directly backs Recommendation 1 in A4 — computed correctly per person, not accidentally double-counted (a real bug we caught and fixed).

In [15]:
fail_rate, n_by_tier, rc_attempt_rate = common.rc_fail_rate_by_device_tier(doc_events_mature, master_mature)
print("RC first-attempt fail rate by device tier:")
print(fail_rate.round(3))
print(f"\nRC attempt rate among mature signups: {rc_attempt_rate:.1%}")
print(f"(i.e. only {rc_attempt_rate:.1%} of signups ever attempt RC at all -- this matters")
print("for correctly scaling any monthly-volume projection off this rate; see A4 Rec 1.)")

RC first-attempt fail rate by device tier:
device_tier
high    0.215
low     0.438
mid     0.292
dtype: float64

RC attempt rate among mature signups: 77.9%
(i.e. only 77.9% of signups ever attempt RC at all -- this matters
for correctly scaling any monthly-volume projection off this rate; see A4 Rec 1.)


### The headline: A2O and R2A
The two numbers the brief says matter most, in that order: the share of sign-ups who get approved (**A2O**), and of those, the share who ever take a ride (**R2A**).

In [16]:
approved_mature = master_mature[master_mature["final_status"] == "approved"]
a2o = len(approved_mature) / len(master_mature)
r2a_given_approved = approved_mature["ever_activated"].mean()
r2a_from_signup = a2o * r2a_given_approved

print(f"A2O  (signup -> approved):                 {a2o:.1%}   ({len(approved_mature):,} / {len(master_mature):,})")
print(f"R2A | Approved (approved -> first order):   {r2a_given_approved:.1%}")
print(f"R2A  (signup -> first order, full chain):   {r2a_from_signup:.1%}")
print()
print("~1 in 6 signups gets approved. Once approved, activation is nearly automatic --")
print("the leak is almost entirely in documents, not post-approval.")

A2O  (signup -> approved):                 17.5%   (3,532 / 20,207)
R2A | Approved (approved -> first order):   98.7%
R2A  (signup -> first order, full chain):   17.2%

~1 in 6 signups gets approved. Once approved, activation is nearly automatic --
the leak is almost entirely in documents, not post-approval.
